In [7]:
"""
transform_data.py
=================
Шаг 2 ETL: Читаем raw_gold_data.csv, считаем производные фичи,
чистим данные, разбиваем на 5 датафреймов под 5 таблиц MySQL.
Результат: 5 CSV файлов готовых к загрузке.
"""

import pandas as pd
import numpy as np

INPUT_FILE = "raw_gold_data.csv"

# ─────────────────────────────────────────────────────────────────────────────

def load_raw() -> pd.DataFrame:
    """Читает сырой CSV и восстанавливает индекс."""
    df = pd.read_csv(INPUT_FILE, index_col="Date", parse_dates=True)
    print(f"  Загружено: {len(df)} строк, {df.shape[1]} колонок")
    return df


In [8]:
def fill_macro_gaps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Макро данные FRED публикуются раз в месяц/квартал.
    Forward fill переносит последнее известное значение на следующие дни
    (стандартная практика для временных рядов с разной частотой).
    """
    macro_cols = [
        "Fed_Rate", "Treasury_10Y", "Treasury_2Y",
        "CPI_Index", "Unemployment", "M2_Supply", "PPI"
    ]
    for col in macro_cols:
        if col in df.columns:
            df[col] = df[col].ffill()
    return df

In [9]:
def compute_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Считает все производные фичи."""

    # ── macro_indicators ──────────────────────────────────────────────────────

    # Инфляция год к году: изменение CPI за 12 месяцев
    # pct_change(252) потому что данные ежедневные (~252 торговых дня в году)
    if "CPI_Index" in df.columns:
        df["Inflation_YoY"] = df["CPI_Index"].ffill().pct_change(252) * 100

    # Реальная ставка по формуле Фишера: номинальная − инфляция
    if "Treasury_10Y" in df.columns and "Inflation_YoY" in df.columns:
        df["Real_Rate_Calc"] = df["Treasury_10Y"] - df["Inflation_YoY"]

    # Кривая доходности: спред 10Y − 2Y (отрицательный = инверсия = сигнал рецессии)
    if "Treasury_10Y" in df.columns and "Treasury_2Y" in df.columns:
        df["Yield_Curve_10Y2Y"] = df["Treasury_10Y"] - df["Treasury_2Y"]

    # ── etf_flows ─────────────────────────────────────────────────────────────

    # Премия GLD к спот-цене золота: насколько ETF дороже/дешевле физического золота
    if "GLD_Price" in df.columns and "Gold_Price" in df.columns:
        df["GLD_Gold_Premium"] = (df["GLD_Price"] / df["Gold_Price"] - 1) * 100

    # Сигнал притока в ETF: объём выше 20-дневной средней = институционалы покупают
    # Используем GLD_Price как прокси (объём не тянем отдельно для простоты)
    if "GLD_Price" in df.columns:
        rolling_avg = df["GLD_Price"].rolling(20).mean()
        df["ETF_Signal"] = (df["GLD_Price"] > rolling_avg).astype(int)

    # ── energy_mining ─────────────────────────────────────────────────────────

    # Индекс себестоимости добычи (AISC proxy):
    # 50% медь (оборудование/электроэнергия) + 30% WTI (дизель) + 20% PPI (инфляция затрат)
    if all(col in df.columns for col in ["Copper_Price", "WTI_Oil", "PPI"]):
        df["Mining_Cost_Index"] = (
            0.50 * df["Copper_Price"] +
            0.30 * df["WTI_Oil"] +
            0.20 * df["PPI"] / 100
        )

        # Маржа добычи: насколько цена золота превышает себестоимость
        df["Gold_Mining_Margin"] = df["Gold_Price"] / df["Mining_Cost_Index"]

        # Бычий сигнал: маржа > 1.8x исторически означает высокую рентабельность
        df["Mining_Bull_Signal"] = (df["Gold_Mining_Margin"] > 1.8).astype(int)

    return df

In [10]:
def build_daily_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Таблица daily_prices: ежедневные цены всех активов кроме ETF и energy.
    Переводим в длинный формат (одна строка = один актив + одна дата).
    """
    price_cols = [
        "Gold_Price", "Silver_Price", "Platinum_Price", "Palladium_Price",
        "DXY_Index", "SP500", "VIX_Index",
        "USD_EUR", "USD_CNY", "USD_RUB",
    ]
    price_cols = [c for c in price_cols if c in df.columns]

    # Маппинг: название колонки → тикер для таблицы assets
    col_to_symbol = {
        "Gold_Price":      "GC=F",
        "Silver_Price":    "SI=F",
        "Platinum_Price":  "PL=F",
        "Palladium_Price": "PA=F",
        "DXY_Index":       "DX-Y.NYB",
        "SP500":           "^GSPC",
        "VIX_Index":       "^VIX",
        "USD_EUR":         "EURUSD=X",
        "USD_CNY":         "USDCNY=X",
        "USD_RUB":         "RUB=X",
    }

    rows = []
    for col in price_cols:
        symbol = col_to_symbol[col]
        tmp = df[[col]].dropna().reset_index()
        tmp.columns = ["price_date", "close_price"]
        tmp["symbol"] = symbol
        rows.append(tmp)

    result = pd.concat(rows, ignore_index=True)
    result = result[["price_date", "symbol", "close_price"]]
    result = result.dropna(subset=["close_price"])
    result = result.sort_values(["price_date", "symbol"]).reset_index(drop=True)

    print(f"  daily_prices: {len(result)} строк")
    return result

In [11]:
def build_macro_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Таблица macro_indicators: макроданные + производные фичи.
    Один ряд на дату (широкий формат).
    Берём только строки где есть хотя бы Fed_Rate.
    """
    cols = [
        "Fed_Rate", "Treasury_10Y", "Treasury_2Y",
        "CPI_Index", "Unemployment", "M2_Supply",
        "Inflation_YoY", "Real_Rate_Calc", "Yield_Curve_10Y2Y",
    ]
    cols = [c for c in cols if c in df.columns]

    result = df[cols].copy()
    result = result.dropna(subset=["Fed_Rate"])
    result.index.name = "indicator_date"
    result = result.reset_index()
    result = result.sort_values("indicator_date").reset_index(drop=True)

    print(f"  macro_indicators: {len(result)} строк")
    return result

In [12]:
def build_etf_flows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Таблица etf_flows: данные ETF фондов GLD и IAU.
    Переводим в длинный формат (одна строка = один ETF + одна дата).
    """
    etf_map = {
        "GLD_Price": "GLD",
        "IAU_Price": "IAU",
    }

    rows = []
    for col, symbol in etf_map.items():
        if col not in df.columns:
            continue
        tmp = df[[col]].dropna().reset_index()
        tmp.columns = ["flow_date", "etf_price"]
        tmp["symbol"] = symbol

        # Добавляем производные фичи только для GLD
        if symbol == "GLD" and "GLD_Gold_Premium" in df.columns:
            tmp = tmp.merge(
                df[["GLD_Gold_Premium", "ETF_Signal"]].reset_index().rename(columns={"Date": "flow_date"}),
                on="flow_date", how="left"
            )
        else:
            tmp["GLD_Gold_Premium"] = None
            tmp["ETF_Signal"] = None

        rows.append(tmp)

    result = pd.concat(rows, ignore_index=True)
    result = result[["flow_date", "symbol", "etf_price", "GLD_Gold_Premium", "ETF_Signal"]]
    result.columns = ["flow_date", "symbol", "etf_price", "gld_gold_premium", "etf_signal"]
    result["etf_signal"] = result["etf_signal"].fillna(0).astype(int)
    result["gld_gold_premium"] = result["gld_gold_premium"].fillna(0)
    result = result.sort_values(["flow_date", "symbol"]).reset_index(drop=True)

    print(f"  etf_flows: {len(result)} строк")
    return result

In [13]:
def build_energy_mining(df: pd.DataFrame) -> pd.DataFrame:
    """
    Таблица energy_mining: нефть, медь, себестоимость добычи.
    Один ряд на дату.
    """
    cols = [
        "WTI_Oil", "Brent_Oil", "Copper_Price",
        "Mining_Cost_Index", "Gold_Mining_Margin", "Mining_Bull_Signal",
    ]
    cols = [c for c in cols if c in df.columns]

    result = df[cols].copy()
    result = result.dropna(subset=["WTI_Oil"])
    result = result.dropna(subset=["Mining_Cost_Index"])
    result = result.dropna(subset=["Gold_Mining_Margin"])
    result.index.name = "record_date"
    result = result.reset_index()
    result = result.sort_values("record_date").reset_index(drop=True)

    print(f"  energy_mining: {len(result)} строк")
    return result

In [14]:
def transform_data():
    print("=== transform_data.py: трансформация данных ===\n")

    df = load_raw()

    print("\nЗаполняем пропуски в макро данных (ffill)...")
    df = fill_macro_gaps(df)

    print("Считаем производные фичи...")
    df = compute_derived_features(df)

    print("\nСтроим датафреймы для каждой таблицы:")
    daily_prices     = build_daily_prices(df)
    macro_indicators = build_macro_indicators(df)
    etf_flows        = build_etf_flows(df)
    energy_mining    = build_energy_mining(df)

    print("\nСохраняем CSV файлы...")
    daily_prices.to_csv("table_daily_prices.csv", index=False)
    macro_indicators.to_csv("table_macro_indicators.csv", index=False)
    etf_flows.to_csv("table_etf_flows.csv", index=False)
    energy_mining.to_csv("table_energy_mining.csv", index=False)

    print("\n=== Готово ===")
    print("  table_daily_prices.csv")
    print("  table_macro_indicators.csv")
    print("  table_etf_flows.csv")
    print("  table_energy_mining.csv")
    print("\nСледующий шаг: запусти load_to_mysql.py")


transform_data()

=== transform_data.py: трансформация данных ===

  Загружено: 6947 строк, 22 колонок

Заполняем пропуски в макро данных (ffill)...
Считаем производные фичи...

Строим датафреймы для каждой таблицы:
  daily_prices: 62459 строк
  macro_indicators: 6947 строк
  etf_flows: 10714 строк
  energy_mining: 6425 строк

Сохраняем CSV файлы...

=== Готово ===
  table_daily_prices.csv
  table_macro_indicators.csv
  table_etf_flows.csv
  table_energy_mining.csv

Следующий шаг: запусти load_to_mysql.py
